In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import col, date_format, to_date, date_format, weekofyear, dayofweek, lit
from pyspark.sql.types import IntegerType

# Build a DataFrame of dates from 2026-03-01 to 2026-03-31
start_date = "2026-03-01"
end_date = "2026-03-31"

dates = spark.range(31).select(
    to_date(lit(start_date)).cast("date").alias("d")
)

from pyspark.sql.functions import expr

dim_date = (
    spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as d_array")
    .select(expr("explode(d_array)").alias("calendar_date"))
    .select(
        date_format(col("calendar_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
        col("calendar_date").cast("date").alias("calendar_date"),
        date_format(col("calendar_date"), "EEEE").alias("day_name"),
        weekofyear(col("calendar_date")).cast(IntegerType()).alias("iso_week"),
        expr("dayofweek(calendar_date) IN (1, 7)").alias("is_weekend"),
    )
)

(dim_date.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_date"))

display(spark.table("gold.dim_date").orderBy("date_key"))

In [0]:
from pyspark.sql.functions import *

dim_store = (
    spark.read.table(f"{catalog}.silver.stores")
    .select(
        col("store_id"),
        col("store_name"),
        col("city"),
        col("state"),
        col("region"),
        col("opening_date"),
    )
)

(dim_store.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_store"))

In [0]:
from pyspark.sql.functions import *

dim_products = (
    spark.read.table("fk_ce1366.silver.products")
    .select(
        col("product_id"),
        col("product_name"),
        col("brand"),
        col("category"),
        col("pack"),
        col("mrp"),
        col("selling_price"),
        col("is_active"),
        col("snapshot_date"),
    )
    .dropDuplicates(["product_id"])
)

(dim_products.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fk_ce1366.gold.dim_products"))

In [0]:
from pyspark.sql.functions import *

dim_customers = (
    spark.read.table("fk_ce1366.silver.customers")
    .select(
        col("customer_sk").cast("bigint"),
        col("customer_id"),
        col("full_name"),
        col("email"),
        col("phone"),
        col("city"),
        col("tier"),
        col("signup_date").cast("date"),
    )
)

(dim_customers.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fk_ce1366.gold.dim_customers"))

In [0]:
from pyspark.sql.functions import *

fact_orders = (
    spark.read.table("fk_ce1366.silver.orders")
    .select(
        col("order_id"),
        col("order_date_key"),
        col("customer_sk"),
        col("customer_id"),
        col("store_id"),
        col("order_status"),
        col("payment_method"),
        col("units"),
        col("gross_amount"),
        col("discount_amount"),
        col("net_item_amount"),
        col("delivery_fee"),
        col("order_total"),
        col("promised_mins"),
        col("actual_mins"),
        col("is_on_time"),
    )
)

(fact_orders.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fk_ce1366.gold.fact_orders"))

In [0]:
from pyspark.sql.functions import *

agg_daily_store_sales = (
    spark.read.table("fk_ce1366.gold.fact_orders")
    .join(
        spark.read.table("fk_ce1366.gold.dim_stores"),
        on="store_id",
        how="left"
    )
    .groupBy(
        col("order_date_key").alias("order_date"),
        col("store_id"),
        col("city")
    )
    .agg(
        sum(when(col("order_status") == "delivered", 1).otherwise(0)).alias("delivered_orders"),
        sum(when(col("order_status") == "cancelled", 1).otherwise(0)).alias("cancelled_orders"),
        sum(col("units")).alias("unit_sold"),
        sum(col("gross_amount")).alias("gross_amount"),
        sum(col("discount_amount")).alias("discount_amount"),
        sum(col("net_item_amount")).alias("net_item_revenue"),
        sum(col("delivery_fee")).alias("delivery_fee_revenue"),
        sum(when(col("is_on_time") == True, 1).otherwise(0)).alias("on_time_orders"),
        (
            sum(when(col("is_on_time") == True, 1).otherwise(0)) /
            count("*") * 100
        ).alias("on_time_rate_pct")
    )
)

(agg_daily_store_sales.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fk_ce1366.gold.agg_daily_store_sales"))